# Gold - Dimensão Vendedores

Cadastro de vendedores com cruzamento dos canais comerciais.

In [ ]:
%run ../0_Config/0-Init

In [ ]:
# Parâmetros de Inicialização
table_name = 'dim_vendedores'
output_path_data = f"{var_gold}/{table_name}/data"
table_name_schema = f'{var_environment}.{var_gold_schema}.{table_name}'

In [ ]:
from pyspark.sql.functions import col, sha2, coalesce, lit

df_vend = spark.read.table(f"{var_environment}.{var_silver_schema}.case_vendedores")
df_canais = spark.read.table(f"{var_environment}.{var_silver_schema}.case_comercial_canais")

df_gold = (
    df_vend.join(df_canais, df_vend.id_canal == df_canais.id_canal, "left")
    .withColumn("sk_vendedor", sha2(col("id_vendedor").cast("string"), 256))
    .select(
        col("sk_vendedor").cast("string").alias("sk_vendedor"),
        col("id_vendedor").cast("integer").alias("id_vendedor"),
        coalesce(col("nome_vendedor"), lit("Vendedor Não Cadastrado")).cast("string").alias("nome_vendedor"),
        coalesce(col("nome_canal"), lit("Sem Canal")).cast("string").alias("nome_canal"),
        coalesce(col("email_vendedor"), lit("Sem E-mail")).cast("string").alias("email_vendedor")
    )
)

In [ ]:
process_data(
    df_write=df_gold,
    tipo_carga='delta',
    nome_gravacao_tabela=table_name_schema,
    caminho_gravacao_tabela=output_path_data,
    chave_clusterby=['nome_canal'],
    chave_upsert='sk_vendedor'
)